In [189]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [190]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


--2026-10-12 14:30:16--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.5’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.03s   

2026-10-12 14:30:16 (7.78 MB/s) - ‘course_lead_scoring_2026.csv.5’ saved [278746/278746]



In [191]:
df = pd.read_csv("course_lead_scoring_2026.csv")
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [192]:
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [193]:
categorical_columns = list(df.select_dtypes(include=['object']).columns)
print("Categorical columns:")
print(categorical_columns)
numerical_columns = list(df.select_dtypes(include=['int64', 'float64']).columns)
numerical_columns = [col for col in numerical_columns if col != 'converted']  # Exclude the target variable
print("Numerical columns:")
print(numerical_columns)

Categorical columns:
['lead_source', 'industry', 'employment_status', 'location']
Numerical columns:
['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


/var/folders/gg/whpf76fd5ysd8_7q0_zzr1l80000gn/T/ipykernel_75813/3796736115.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = list(df.select_dtypes(include=['object']).columns)


In [194]:
for col in categorical_columns:
    df[col] = df[col].fillna('NA')
    
df['annual_income'] = df['annual_income'].fillna(0.0)
df['lead_score'] = df['lead_score'].fillna(0.0)

In [195]:
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [196]:
print(df['industry'].value_counts())

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


In [197]:
df.converted

0       1
1       1
2       1
3       0
4       1
       ..
4995    1
4996    0
4997    1
4998    1
4999    1
Name: converted, Length: 5000, dtype: int64

In [198]:
correlation_matrix = df.corr(numeric_only=True)
print("Correlation matrix:")
print(correlation_matrix)

Correlation matrix:
                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   
converted                      0.189652                  0.395038   

                          interaction_count  lead_score  converted  
annual_income                      0.122842    0.229496   0.189652  
number_of_courses_viewed           0.721609    0.757204   0.395038  
interaction_count                  1.000000    0.915746   0.448624  
lead_score                         0.915746    1.000000   0.483246  
converted                          0.448624    0.483246   1.000000  


In [199]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [200]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [201]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [202]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [203]:
df_full_train = df_full_train.reset_index(drop=True)

In [204]:
df_full_train.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [205]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64

In [206]:
print("Industry:", mutual_info_score(df_full_train['converted'], df_full_train['industry']))
print("Location:", mutual_info_score(df_full_train['converted'], df_full_train['location']))    
print("Lead Source:", mutual_info_score(df_full_train['converted'], df_full_train['lead_source']))
print("Employment Status:", mutual_info_score(df_full_train['converted'], df_full_train['employment_status']))

Industry: 0.0028984818739526547
Location: 0.0010498214288891111
Lead Source: 0.0347609795100606
Employment Status: 0.02031283511067701


In [207]:
def mutual_info_score_categorical(series):
    return mutual_info_score(series, df_full_train['converted'])

df_full_train[categorical_columns].apply(mutual_info_score_categorical)

lead_source          0.034761
industry             0.002898
employment_status    0.020313
location             0.001050
dtype: float64

In [208]:
train_dicts = df_train[categorical_columns + numerical_columns].to_dict(orient='records')

In [209]:
dv = DictVectorizer(sparse=False)

In [210]:
X_train = dv.fit_transform(train_dicts)

In [211]:
val_dicts = df_val[categorical_columns + numerical_columns].to_dict(orient='records')

In [212]:
X_val = dv.transform(val_dicts)

In [213]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [214]:
model.coef_.round(3)

array([[-0.   , -0.001,  0.007, -0.003, -0.008, -0.006, -0.   , -0.003,
        -0.001, -0.002, -0.002, -0.002, -0.001,  0.137,  0.011,  0.   ,
        -0.009,  0.007, -0.014,  0.009, -0.004,  0.   , -0.002, -0.002,
        -0.001, -0.003, -0.004,  0.054]])

In [215]:
model.intercept_

array([-0.01083898])

In [216]:
y_pred = model.predict_proba(X_val)[:, 1]

In [217]:
converted_decision = (y_pred >= 0.5).astype(int)

In [218]:
(y_val == converted_decision).mean()

np.float64(0.645)

In [219]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision
df_pred['actual'] = y_val

In [220]:
df_pred['correct'] = df_pred['prediction'] == df_pred['actual']

In [221]:
df_pred

,probability,prediction,actual,correct
0,0.617257,1,1,True
1,0.443641,0,1,False
2,0.604340,1,0,False
3,0.645622,1,1,True
4,0.522142,1,0,False
...,...,...,...,...
995,0.684710,1,1,True
996,0.669580,1,1,True
997,0.458473,0,0,True
998,0.561770,1,0,False


In [225]:
round(df_pred.correct.mean(), 2)

np.float64(0.64)

In [231]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

{'annual_income': np.float64(-0.0),
 'employment_status=NA': np.float64(-0.001),
 'employment_status=employed': np.float64(0.007),
 'employment_status=self_employed': np.float64(-0.003),
 'employment_status=student': np.float64(-0.008),
 'employment_status=unemployed': np.float64(-0.006),
 'industry=NA': np.float64(-0.0),
 'industry=education': np.float64(-0.003),
 'industry=finance': np.float64(-0.001),
 'industry=healthcare': np.float64(-0.002),
 'industry=manufacturing': np.float64(-0.002),
 'industry=retail': np.float64(-0.002),
 'industry=technology': np.float64(-0.001),
 'interaction_count': np.float64(0.137),
 'lead_score': np.float64(0.011),
 'lead_source=NA': np.float64(0.0),
 'lead_source=events': np.float64(-0.009),
 'lead_source=organic_search': np.float64(0.007),
 'lead_source=paid_ads': np.float64(-0.014),
 'lead_source=referral': np.float64(0.009),
 'lead_source=social_media': np.float64(-0.004),
 'location=NA': np.float64(0.0),
 'location=africa': np.float64(-0.002),
 '

In [232]:
features = categorical_columns + numerical_columns

In [233]:
def train_accuracy(cols):
    dv_ = DictVectorizer(sparse=False)
    X_tr = dv_.fit_transform(df_train[cols].to_dict(orient='records'))
    X_v = dv_.transform(df_val[cols].to_dict(orient='records'))
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    return (y_val == (m.predict_proba(X_v)[:, 1] >= 0.5).astype(int)).mean()

In [234]:
baseline = train_accuracy(features)
print('baseline', baseline, 'Q4', (y_val == converted_decision).mean())

baseline 0.645 Q4 0.645


In [238]:
diffs = {f: baseline - train_accuracy([c for c in features if c != f]) for f in features}
for f, d in sorted(diffs.items(), key=lambda kv: kv[1]):
    print(f, d)

annual_income -0.07899999999999996
industry 0.0
employment_status 0.0
location 0.0
lead_score 0.0010000000000000009
number_of_courses_viewed 0.0020000000000000018
lead_source 0.0030000000000000027
interaction_count 0.04400000000000004


In [239]:
Cs = [0.000001, 0.00001, 0.0001, 0.001]

In [240]:
for C in Cs:
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    acc = (y_val == (m.predict_proba(X_val)[:, 1] >= 0.5).astype(int)).mean()
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
